# Chapter 03. Supervised & Unsupervised Learning

Supervised Learning의 Classification과 Unsupervised Learning의 Clustering을 비교합니다. 주요 실습 데이터는 Notebook 안에서 생성하므로 별도 CSV 업로드가 필요하지 않습니다.

프로젝트별 구현 STEP과 모범 구현은 Chapter 03의 `README.md` 및 `projects/` 디렉터리에 있습니다.

## 1. 실행 환경

Google Colab 또는 로컬 Jupyter Notebook에서 위에서 아래로 실행합니다. CPU로 실행할 수 있습니다.

In [ ]:
# 필수 패키지가 설치되지 않은 환경에서만 사용합니다.
# %pip install numpy pandas matplotlib scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

print("Chapter 03: environment ready")

## 2. Classification과 Logistic Regression

이진 Logistic Regression은 선형 Score에 Sigmoid를 적용하여 클래스 1의 확률 추정치를 계산합니다. 실제 Label을 결정할 때는 Decision Threshold를 사용합니다.

In [ ]:
z = np.array([-2., 0., 2.])
probability = 1 / (1 + np.exp(-z))
print("Sigmoid:", np.round(probability, 3))
print("Threshold=0.5:", (probability >= 0.5).astype(int))

## 3. 문의 긴급 Label 예측

합성 데이터 480건을 생성하고 Train/Test를 75%/25%로 분리합니다. 이 생성 과정은 프로젝트 CSV와 같은 시드·수식을 사용합니다.

In [ ]:
rng = np.random.default_rng(303)
n = 480
waiting_hours = np.round(rng.uniform(0.25, 20.0, n), 2)
failed_attempts = rng.poisson(1.7, n).clip(0, 9)
impacted_users = rng.integers(1, 150, n)
score = 2.5 * (-3.5 + 0.17 * waiting_hours + 0.80 * failed_attempts + 0.009 * impacted_users)
urgent = rng.binomial(1, 1 / (1 + np.exp(-score)))
X_tickets = np.column_stack([waiting_hours, failed_attempts, impacted_users])
X_train, X_test, y_train, y_test = train_test_split(
    X_tickets, urgent, test_size=0.25, random_state=42, stratify=urgent
)
classifier = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
classifier.fit(X_train, y_train)
labels_pred = classifier.predict(X_test)
prob_urgent = classifier.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, labels_pred, labels=[0, 1])
print(f"Accuracy: {accuracy_score(y_test, labels_pred):.1%}")
print("Confusion matrix:\n", cm)
print("Probabilities for first five test samples:", np.round(prob_urgent[:5], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Normal", "Urgent"],
       yticklabels=["Normal", "Urgent"], xlabel="Predicted", ylabel="Actual",
       title="Service ticket classification")
plt.tight_layout()
plt.show()

## 4. KMeans의 중심 갱신

정답 Label 없이 Sample을 가까운 중심에 할당합니다. 다음 코드는 전체 KMeans가 아니라 할당과 평균 계산을 한 번 수행한 결과입니다.

In [ ]:
points = np.array([[1., 1.], [1., 2.], [2., 1.], [7., 7.], [8., 7.], [7., 8.]])
centers = np.array([[1., 1.], [8., 8.]])
distance = np.linalg.norm(points[:, None, :] - centers[None, :, :], axis=2)
assigned = np.argmin(distance, axis=1)
updated_centers = np.vstack([points[assigned == i].mean(axis=0) for i in range(2)])
print("Cluster IDs:", assigned)
print("Updated centers:\n", updated_centers)

## 5. 대여소 이용 패턴 Clustering

대여소 데이터에는 Target이 없습니다. 세 Feature를 표준화해 군집화하고, Cluster별 평균은 원래 이용량 단위로 계산합니다.

In [ ]:
rng = np.random.default_rng(304)
centers = np.array([[78, 72, 25], [25, 30, 86], [50, 54, 52]], dtype=float)
groups = np.repeat(np.arange(3), 90)
rng.shuffle(groups)
station_values = centers[groups] + rng.normal(0, [10, 11, 10], (len(groups), 3))
station_values = np.round(np.maximum(station_values, 0), 1)
station_df = pd.DataFrame(station_values, columns=["morning_trips", "evening_trips", "weekend_trips"])
station_model = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=42))
station_clusters = station_model.fit_predict(station_df)
station_df["cluster"] = station_clusters
print(station_df.groupby("cluster")[["morning_trips", "evening_trips", "weekend_trips"]].mean().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sc = ax.scatter(station_df["morning_trips"], station_df["weekend_trips"], c=station_clusters, cmap="viridis", s=18)
ax.set(xlabel="Morning trips", ylabel="Weekend trips", title="Station clusters")
fig.colorbar(sc, ax=ax, label="Cluster ID")
plt.tight_layout()
plt.show()

## 6. Label과 Cluster ID는 다릅니다

활동 유형을 예측하는 Classification에는 Target을 학습 데이터로 전달하지만, KMeans는 Feature만 사용합니다. ARI는 군집화 이후 참고 Label을 이용하여 계산합니다.

In [ ]:
rng = np.random.default_rng(305)
n = 540
morning = rng.normal(40, 14, n).clip(5, 85)
remote = rng.normal(34, 15, n).clip(1, 85)
evening = rng.normal(27, 12, n).clip(1, 70)
score = 0.085 * morning - 0.075 * remote + 0.055 * evening + rng.normal(0, 0.7, n)
boundaries = np.quantile(score, [1/3, 2/3])
work_label = np.digitize(score, boundaries)
X_work = np.column_stack([np.round(morning, 2), np.round(remote, 2), np.round(evening, 2)])
Xtr, Xte, ytr, yte = train_test_split(X_work, work_label, test_size=0.3, stratify=work_label, random_state=42)
work_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)).fit(Xtr, ytr)
work_cls = work_clf.predict(Xte)
work_clusters = make_pipeline(StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=42)).fit_predict(Xte)
print(f"Classification Accuracy: {accuracy_score(yte, work_cls):.1%}")
print(f"Clustering ARI: {adjusted_rand_score(yte, work_clusters):.3f}")

## 7. KMeans와 DBSCAN: 곡선형 군집 및 Noise

같은 좌표를 KMeans와 DBSCAN으로 분석합니다. `reference_group`은 학습에 사용하지 않고 **사후 ARI 계산**에만 사용합니다.

In [ ]:
rng = np.random.default_rng(306)
moons, moon_labels = make_moons(n_samples=440, noise=0.075, random_state=306)
outliers = rng.uniform(low=[-1.5, -1.1], high=[2.6, 1.7], size=(24, 2))
X_paths = np.vstack([moons, outliers])
reference = np.concatenate([moon_labels, -np.ones(len(outliers), dtype=int)])
order = rng.permutation(len(X_paths))
X_paths, reference = X_paths[order], reference[order]
# CSV와 동일하게 소수점 다섯 자리까지 저장한 좌표를 사용합니다.
X_paths = np.round(X_paths, 5)
scaled_paths = StandardScaler().fit_transform(X_paths)
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(scaled_paths)
db = DBSCAN(eps=0.23, min_samples=6).fit_predict(scaled_paths)
print(f"KMeans ARI: {adjusted_rand_score(reference, km):.3f}")
print(f"DBSCAN ARI: {adjusted_rand_score(reference, db):.3f}")
print("DBSCAN clusters:", len(set(db) - {-1}), "noise:", (db == -1).sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, y, title in zip(axes, [reference, km, db], ["Reference (evaluation only)", "KMeans", "DBSCAN (-1: noise)"]):
    ax.scatter(X_paths[:, 0], X_paths[:, 1], c=y, cmap="viridis", s=13)
    ax.set(xlabel="x position", ylabel="y position", title=title)
    ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## 8. 실습 점검

1. 문의 분류의 Decision Threshold를 변경하고 FP·FN이 어떻게 달라지는지 확인합니다.
2. 대여소 Cluster별 평균을 원래 이용량 단위로 설명합니다.
3. `work_mode`를 KMeans의 입력에 사용하지 않은 이유를 설명합니다.
4. DBSCAN의 `eps`를 변경하고 Noise 수와 Cluster 수를 비교합니다.

각 프로젝트의 전체 구현 요구사항과 정답 코드는 Chapter 03의 `README.md`에 연결되어 있습니다.